# JSON Ingestion Pipeline Demo
## Testing `IngestionService` with `accommodation_halls.json`

This notebook walks through every step of the ingestion pipeline defined in `services/ingestion_service.py`, using the accommodation data as the test input and **OpenAI `text-embedding-3-small`** as the embedder.

The pipeline stops **just before** the MongoDB upsert so you can inspect the final `ChunkRecord` objects.

**Pipeline steps:**
1. Normalize JSON → `Document`
2. Segment `Document` → `Chunk` list
3. Tag each `Chunk` → `ChunkTags`
4. Embed all chunk texts → vectors (OpenAI)
5. Build `ChunkRecord` objects ← _final output before upsert_

## 1. Install & Import Dependencies

In [1]:
%pip install openai python-dotenv pydantic --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import json
import hashlib
from pprint import pprint
from typing import Any, Dict, List, Optional, Union

# --- Make project root importable ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- Import schemas directly (absolute import from project root) ---
from schemas.models import Document, Chunk, ChunkTags, ChunkRecord

# --- OpenAI ---
import openai
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from a .env file if present

# If you don't have a .env file, set your key directly here:
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("✓ Imports OK")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  OpenAI key   : {'set' if os.getenv('OPENAI_API_KEY') else 'NOT SET — add to .env or set above'}")

✓ Imports OK
  Project root : c:\Users\Emman\OneDrive\Documents\GitHub\PartD_GroupProject
  OpenAI key   : set


## 2. Load Accommodation JSON Data

In [3]:
JSON_PATH = os.path.join(PROJECT_ROOT, "data", "accommodation_halls.json")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    accommodation_data = json.load(f)

print(f"✓ Loaded {len(accommodation_data)} halls from {JSON_PATH}\n")
print("=" * 60)
print("First hall entry (full structure):")
print("=" * 60)
pprint(accommodation_data[0], depth=3)

✓ Loaded 17 halls from c:\Users\Emman\OneDrive\Documents\GitHub\PartD_GroupProject\data\accommodation_halls.json

First hall entry (full structure):
{'address': 'Butler Court Loughborough University LE11 3TS',
 'catering_type': 'self_catered',
 'contact_email': 'sac@mailbox.lboro.ac.uk',
 'contact_phone': '+44 01509 274488',
 'contacts': {'duty_mobile': '07731 990251',
              'general_enquiries_email': 'eastparkreception@lboro.ac.uk',
              'general_enquiries_phone': '+44 (0)1509 223819',
              'hall_manager': {'name': 'Sarah Hooley',
                               'phone': '+44 (0)1509 223819'},
              'subwardens': [{...}, {...}, {...}],
              'warden': {'email': 'butlercourt.warden@lboro.ac.uk',
                         'name': 'Email:',
                         'phone': None}},
 'days_catered': None,
 'facilities': ['Deluxe flats have sofa and TV in the kitchen',
                'Electric hobs in the kitchen',
                'Large games/commo

## 3. Initialize OpenAI Embedder

A lightweight wrapper that mirrors the `embed_many(texts)` interface expected by `IngestionService`.

In [4]:
class OpenAIEmbedder:
    """
    Minimal embedder wrapper compatible with IngestionService.embed_many(texts).
    Uses OpenAI text-embedding-3-small (1536 dims).
    """

    def __init__(self, model: str = "text-embedding-3-small"):
        self.model = model
        self.client = openai.OpenAI()  # reads OPENAI_API_KEY from env

    def embed_many(self, texts: List[str]) -> List[List[float]]:
        """Batch embed a list of strings. Returns a list of float vectors."""
        if not texts:
            return []
        response = self.client.embeddings.create(
            model=self.model,
            input=texts,
        )
        # response.data is sorted by index
        return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]


# ---- Smoke test ----
embedder = OpenAIEmbedder()
test_vecs = embedder.embed_many(["Butler Court is a self-catered hall at Loughborough."])

print(f"✓ Embedder OK")
print(f"  Model         : {embedder.model}")
print(f"  Vectors returned : {len(test_vecs)}")
print(f"  Embedding dim    : {len(test_vecs[0])}")
print(f"  First 5 values   : {test_vecs[0][:5]}")

✓ Embedder OK
  Model         : text-embedding-3-small
  Vectors returned : 1
  Embedding dim    : 1536
  First 5 values   : [-0.04096465930342674, -0.01874755322933197, 0.006059754639863968, -0.010576155968010426, -0.03747117891907692]


## 4. Initialize Ingestion Service

We use `repo=None` because we stop before the upsert step. `tagger=None` means the heuristic tagger will be used.

In [5]:
# IngestionService uses relative imports (from ..schemas.models) which can't be
# used directly in a notebook. We inline an equivalent copy here that uses the
# absolute imports already resolved above.

class IngestionService:
    """
    Offline pipeline (notebook-compatible copy):
    input -> Document -> Chunks -> Tags -> Embeddings -> [MongoDB upsert — skipped in this demo]
    """

    def __init__(self, repo, embedder, tagger=None, chunker=None, version: str = "v1"):
        self.repo = repo
        self.embedder = embedder
        self.tagger = tagger
        self.chunker = chunker
        self.version = version

    # ---- Public entrypoints ----

    def ingest_json(self, data, source_id: str, title: Optional[str] = None) -> int:
        doc = self._normalize_json(data=data, source_id=source_id, title=title)
        return self._ingest_document(doc)

    # ---- Core pipeline ----

    def _ingest_document(self, doc: Document) -> int:
        chunks = self._segment(doc)
        tagged = [(c, self._tag_chunk(doc, c)) for c in chunks]
        texts = [c.text for (c, _) in tagged]
        vectors = self.embedder.embed_many(texts)
        records = [self._build_record(doc, chunk, tags, vec)
                   for (chunk, tags), vec in zip(tagged, vectors)]
        # ---- UPSERT SKIPPED IN THIS DEMO ----
        # self.repo.upsert_chunks(records)
        return records  # return records instead of count for inspection

    # ---- Normalizers ----

    def _normalize_json(self, data, source_id: str, title: Optional[str]) -> Document:
        text = self._json_to_text(data)
        return Document(source_id=source_id, source_type="json", title=title, text=text, raw={"json": data})

    def _json_to_text(self, obj: Any, prefix: str = "") -> str:
        lines: List[str] = []

        def walk(x: Any, path: str):
            if isinstance(x, dict):
                for k, v in x.items():
                    walk(v, f"{path}.{k}" if path else k)
            elif isinstance(x, list):
                for i, v in enumerate(x):
                    walk(v, f"{path}[{i}]")
            else:
                val = str(x).strip()
                if val:
                    lines.append(f"{path}: {val}")

        walk(obj, prefix)
        return "\n".join(lines)

    # ---- Segmenter ----

    def _segment(self, doc: Document) -> List[Chunk]:
        if doc.source_type == "web":
            blocks = [b.strip() for b in doc.text.split("---") if b.strip()]
            return [self._make_chunk(doc.source_id, b, order=i, section="qna") for i, b in enumerate(blocks)]
        if doc.source_type == "json":
            return self._line_group_chunk(doc.source_id, doc.text, group_size=30, section="json_fields")
        return self._line_group_chunk(doc.source_id, doc.text, group_size=40, section="text")

    def _line_group_chunk(self, source_id: str, text: str, group_size: int, section: str) -> List[Chunk]:
        lines = [ln for ln in text.splitlines() if ln.strip()]
        chunks: List[Chunk] = []
        for i in range(0, len(lines), group_size):
            block = "\n".join(lines[i: i + group_size])
            chunks.append(self._make_chunk(source_id, block, order=len(chunks), section=section))
        return chunks

    def _make_chunk(self, source_id: str, text: str, order: int, section: Optional[str]) -> Chunk:
        chunk_id = self._stable_id(f"{source_id}:{order}:{text[:80]}")
        return Chunk(chunk_id=chunk_id, source_id=source_id, text=text, order=order, section=section)

    # ---- Tagging ----

    def _tag_chunk(self, doc: Document, chunk: Chunk) -> ChunkTags:
        domain_guess = self._heuristic_domain(doc, chunk)
        if self.tagger:
            return self.tagger.tag(text=chunk.text, hint_domain=domain_guess)
        return ChunkTags(
            domain=domain_guess or "unknown",
            entity_tags=self._heuristic_entities(chunk.text),
            confidence=0.3,
        )

    def _heuristic_domain(self, doc: Document, chunk: Chunk) -> Optional[str]:
        t = chunk.text.lower()
        if "catering" in t or "launderette" in t or "ensuite" in t or "self-catered" in t:
            return "accommodation"
        if "entry requirements" in t or "ucas" in t or "modules" in t:
            return "courses"
        return None

    def _heuristic_entities(self, text: str) -> List[str]:
        return []

    # ---- Record builder ----

    def _build_record(self, doc: Document, chunk: Chunk, tags: ChunkTags, embedding: List[float]) -> ChunkRecord:
        return ChunkRecord(
            chunk_id=chunk.chunk_id,
            source_id=doc.source_id,
            source_type=doc.source_type,
            title=doc.title,
            url=doc.url,
            text=chunk.text,
            embedding=embedding,
            domain=tags.domain,
            entity_tags=tags.entity_tags,
            section=chunk.section,
            order=chunk.order,
            metadata={"key_fields": tags.key_fields, "confidence": tags.confidence},
            version=self.version,
        )

    def _stable_id(self, s: str) -> str:
        return hashlib.sha1(s.encode("utf-8")).hexdigest()


# ---- Instantiate ----
service = IngestionService(
    repo=None,         # not needed — we stop before upsert
    embedder=embedder, # OpenAIEmbedder defined above
    tagger=None,       # use heuristic tagger
    chunker=None,
    version="v1",
)

print("✓ IngestionService ready")
print(f"  embedder : {service.embedder.__class__.__name__} ({service.embedder.model})")
print(f"  tagger   : {service.tagger}  (heuristic fallback active)")
print(f"  repo     : {service.repo}   (upsert will be skipped)")
print(f"  version  : {service.version}")

✓ IngestionService ready
  embedder : OpenAIEmbedder (text-embedding-3-small)
  tagger   : None  (heuristic fallback active)
  repo     : None   (upsert will be skipped)
  version  : v1


---
## Step 1: Normalize JSON → `Document`

`_normalize_json()` flattens the entire JSON structure into a key-path text representation, e.g. `[0].name: Butler Court`, which is stored as `Document.text`.  
The raw JSON is preserved in `Document.raw`.

In [6]:
SOURCE_ID = "accommodation_halls"

# Call Step 1 directly
doc = service._normalize_json(
    data=accommodation_data,
    source_id=SOURCE_ID,
    title="Loughborough Accommodation Halls",
)

print("=== Document Object ===")
print(f"  source_id   : {doc.source_id}")
print(f"  source_type : {doc.source_type}")
print(f"  title       : {doc.title}")
print(f"  raw keys    : {list(doc.raw.keys())}")
print(f"  text length : {len(doc.text)} characters  |  {len(doc.text.splitlines())} lines")
print()
print("=== First 500 characters of Document.text (key-path flattened) ===")
print("-" * 60)
print(doc.text[:500])
print("...")

=== Document Object ===
  source_id   : accommodation_halls
  source_type : json
  title       : Loughborough Accommodation Halls
  raw keys    : ['json']
  text length : 64307 characters  |  1299 lines

=== First 500 characters of Document.text (key-path flattened) ===
------------------------------------------------------------
[0].name: Butler Court
[0].type: hall
[0].address: Butler Court Loughborough University LE11 3TS
[0].short_description: Located in East Park, near the School of Design and Creative Arts and our rugby pitches. Butler Court is a very tight knit community where morale is high and there is a great social life. The hall is named after Sir Clifford Butler, Vice-Chancellor of the University from 1975-1985.
[0].catering_type: self_catered
[0].days_catered: None
[0].meals_per_week: None
[0].services[0]: 
...


---
## Step 2: Segment `Document` → `Chunk` List

`_segment()` groups lines of `Document.text` into windows of 30 lines each (for `source_type="json"`), producing one `Chunk` per window.  
Each `Chunk` gets a stable SHA-1 `chunk_id` derived from its content.

In [7]:
# Call Step 2 directly
chunks = service._segment(doc)

print(f"✓ Segmentation complete")
print(f"  Total chunks produced : {len(chunks)}")
print(f"  group_size            : 30 lines per chunk (json strategy)")
print()

for i, chunk in enumerate(chunks[:3]):
    print(f"{'=' * 60}")
    print(f"Chunk {chunk.order}  |  chunk_id: {chunk.chunk_id}")
    print(f"  section : {chunk.section}")
    print(f"  lines   : {len(chunk.text.splitlines())}")
    print(f"  text preview (first 300 chars):")
    print(f"  {chunk.text[:300].replace(chr(10), chr(10) + '  ')}")
    print()

print(f"... ({len(chunks) - 3} more chunks not shown)")

✓ Segmentation complete
  Total chunks produced : 44
  group_size            : 30 lines per chunk (json strategy)

Chunk 0  |  chunk_id: 50f7658e8e5df42af88d80423c04ea0200678853
  section : json_fields
  lines   : 30
  text preview (first 300 chars):
  [0].name: Butler Court
  [0].type: hall
  [0].address: Butler Court Loughborough University LE11 3TS
  [0].short_description: Located in East Park, near the School of Design and Creative Arts and our rugby pitches. Butler Court is a very tight knit community where morale is high and there is a great socia

Chunk 1  |  chunk_id: 6a41df4efcde5ea7334fbd2912d5b2fd93690cf4
  section : json_fields
  lines   : 30
  text preview (first 300 chars):
  [0].room_types[1].occupancy: double
  [0].room_types[1].tenancy_weeks: 41
  [0].room_types[1].number_of_beds: 204
  [0].room_types[1].prices[0].year: 2025/26
  [0].room_types[1].prices[0].per_week_gbp: 128.16
  [0].room_types[1].prices[0].total_contract_gbp: 5364.41
  [0].room_types[1].available_to[0

---
## Step 3: Tag Each Chunk → `ChunkTags`

`_tag_chunk()` runs heuristic keyword matching to assign a `domain` label (`"accommodation"`, `"courses"`, or `"unknown"`) and `entity_tags`.  
Since `tagger=None`, no LLM is called — only the fast heuristic path is used.

In [8]:
# Call Step 3 for every chunk
tagged_pairs = [(chunk, service._tag_chunk(doc, chunk)) for chunk in chunks]

# Summarise domain distribution
from collections import Counter
domain_counts = Counter(tags.domain for _, tags in tagged_pairs)
print(f"✓ Tagging complete — {len(tagged_pairs)} chunks tagged")
print(f"  Domain distribution: {dict(domain_counts)}")
print()

# Show 5 sample tagged chunks
print("=== Sample: 5 tagged chunks ===")
for chunk, tags in tagged_pairs[:5]:
    print(f"{'-' * 60}")
    print(f"  chunk #{chunk.order}  |  domain: {tags.domain}  |  confidence: {tags.confidence}")
    print(f"  entity_tags : {tags.entity_tags or '[]'}")
    print(f"  text preview: {chunk.text[:200].replace(chr(10), ' | ')}")
    print()

✓ Tagging complete — 44 chunks tagged
  Domain distribution: {'accommodation': 31, 'unknown': 13}

=== Sample: 5 tagged chunks ===
------------------------------------------------------------
  chunk #0  |  domain: accommodation  |  confidence: 0.3
  entity_tags : []
  text preview: [0].name: Butler Court | [0].type: hall | [0].address: Butler Court Loughborough University LE11 3TS | [0].short_description: Located in East Park, near the School of Design and Creative Arts and our rugby 

------------------------------------------------------------
  chunk #1  |  domain: accommodation  |  confidence: 0.3
  entity_tags : []
  text preview: [0].room_types[1].occupancy: double | [0].room_types[1].tenancy_weeks: 41 | [0].room_types[1].number_of_beds: 204 | [0].room_types[1].prices[0].year: 2025/26 | [0].room_types[1].prices[0].per_week_gbp: 128.16

------------------------------------------------------------
  chunk #2  |  domain: unknown  |  confidence: 0.3
  entity_tags : []
  text preview

---
## Step 4: Embed Chunks with OpenAI

All chunk texts are sent to OpenAI's `text-embedding-3-small` model in a **single batched API call**. Each text gets a 1536-dimensional float vector.

In [9]:
texts_to_embed = [chunk.text for chunk, _ in tagged_pairs]

print(f"Sending {len(texts_to_embed)} texts to OpenAI for embedding...")
print(f"(This is a single batched API call)")
print()

# Call Step 4 directly — mirrors the line in _ingest_document
vectors = embedder.embed_many(texts_to_embed)

print(f"✓ Embedding complete")
print(f"  Vectors returned  : {len(vectors)}")
print(f"  Embedding dim     : {len(vectors[0])}")
print()
print("=== Preview: first embedding vector (first 10 values) ===")
print(f"  chunk #0 → [{', '.join(f'{v:.6f}' for v in vectors[0][:10])}, ...]")

Sending 44 texts to OpenAI for embedding...
(This is a single batched API call)

✓ Embedding complete
  Vectors returned  : 44
  Embedding dim     : 1536

=== Preview: first embedding vector (first 10 values) ===
  chunk #0 → [-0.063333, -0.019331, 0.036692, -0.047200, -0.058764, 0.013028, -0.004786, -0.009423, 0.025684, -0.024214, ...]


---
## Step 5: Build Final `ChunkRecord` Objects (Pre-Upsert Inspection)

`_build_record()` assembles every field — text, embedding, tags, metadata — into a `ChunkRecord`.  
This is the **exact object that would be upserted into MongoDB**. The embedding is truncated to 5 values below for readability.

In [10]:
# Call Step 5 for every (chunk, tags, vector) triple
records: List[ChunkRecord] = [
    service._build_record(doc, chunk, tags, vec)
    for (chunk, tags), vec in zip(tagged_pairs, vectors)
]

print(f"✓ {len(records)} ChunkRecord objects built — UPSERT SKIPPED")
print()
print("=" * 60)
print("=== Final ChunkRecord objects (first 3) — ready for MongoDB ===")
print("=" * 60)

for record in records[:3]:
    # Build a display dict with the embedding truncated
    display = record.model_dump()
    display["embedding"] = display["embedding"][:5] + ["... (1536 dims total)"]

    print(f"\n--- Record #{record.order} ---")
    pprint(display, sort_dicts=False)

print()
print(f"... ({len(records) - 3} more records not shown)")
print()
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"  Source file   : accommodation_halls.json ({len(accommodation_data)} halls)")
print(f"  Document text : {len(doc.text)} chars, {len(doc.text.splitlines())} lines")
print(f"  Chunks        : {len(chunks)}")
print(f"  Embedder      : {embedder.model} ({len(vectors[0])} dims)")
print(f"  ChunkRecords  : {len(records)} ready for upsert into MongoDB")
print(f"  MongoDB upsert: SKIPPED in this demo")

✓ 44 ChunkRecord objects built — UPSERT SKIPPED

=== Final ChunkRecord objects (first 3) — ready for MongoDB ===

--- Record #0 ---
{'chunk_id': '50f7658e8e5df42af88d80423c04ea0200678853',
 'source_id': 'accommodation_halls',
 'source_type': 'json',
 'title': 'Loughborough Accommodation Halls',
 'url': None,
 'text': '[0].name: Butler Court\n'
         '[0].type: hall\n'
         '[0].address: Butler Court Loughborough University LE11 3TS\n'
         '[0].short_description: Located in East Park, near the School of '
         'Design and Creative Arts and our rugby pitches. Butler Court is a '
         'very tight knit community where morale is high and there is a great '
         'social life. The hall is named after Sir Clifford Butler, '
         'Vice-Chancellor of the University from 1975-1985.\n'
         '[0].catering_type: self_catered\n'
         '[0].days_catered: None\n'
         '[0].meals_per_week: None\n'
         '[0].services[0]: Fortnightly bedroom cleaning\n'
         